# Big Data Management - University of Waterloo
### Winter 2026
### Group 10


## Team Members
- Rosy Zhou
- Miguel Morales Gonzalez


---

## Windows Environment Setup Guide

This section documents every step required to run this project on a **Windows machine** using a local Spark installation.

---

### Step 1 — Install Java 11 (JDK)

Spark requires Java 11. We used **Eclipse Temurin 11** (formerly AdoptOpenJDK).

**Option A — Install via winget (recommended):**
```powershell
winget install --id EclipseAdoptium.Temurin.11.JDK -e
```

**Option B — Manual install:**
Download from https://adoptium.net and choose **JDK 11, Windows x64**.

After installation, set the user environment variable:
```powershell
$jdk = Get-ChildItem "C:\Program Files\Eclipse Adoptium" -Directory |
  Where-Object { $_.Name -like "jdk-11*" } |
  Sort-Object Name -Descending |
  Select-Object -First 1

[Environment]::SetEnvironmentVariable("JAVA_HOME", $jdk.FullName, "User")

$path = [Environment]::GetEnvironmentVariable("Path", "User")
if ($path -notlike "*%JAVA_HOME%\bin*") {
  [Environment]::SetEnvironmentVariable("Path", "$path;%JAVA_HOME%\bin", "User")
}
```

Verify:
```powershell
java -version
# Expected: openjdk version "11.x.x" ...
```

---

### Step 2 — Install Hadoop Binaries (winutils)

Spark on Windows requires `winutils.exe` and `hadoop.dll` for local filesystem access. Without them, Spark throws native I/O errors when reading/writing Parquet.

1. Download the binaries matching your Spark's bundled Hadoop version.
   For **Spark 3.4.x** → use **Hadoop 3.3.x**.
   Trusted source: https://github.com/cdarlint/winutils

2. Place both files in `C:\hadoop\bin\`:
   ```
   C:\hadoop\bin\winutils.exe
   C:\hadoop\bin\hadoop.dll
   ```

3. Set environment variables (one-time, in PowerShell):
   ```powershell
   [Environment]::SetEnvironmentVariable("HADOOP_HOME", "C:\hadoop", "User")
   $path = [Environment]::GetEnvironmentVariable("Path", "User")
   [Environment]::SetEnvironmentVariable("Path", "$path;C:\hadoop\bin", "User")
   ```

> **Note:** Every notebook and script in this project also sets `HADOOP_HOME` programmatically via `os.environ` before Spark starts, so the pipeline works even if the system variable is not set persistently.

---

### Step 3 — Install Anaconda / Python Environment

Install **Anaconda** (https://www.anaconda.com/download) or use an existing Python 3.9+ environment.

Create and activate a dedicated environment (optional but recommended):
```powershell
conda create -n bigdata python=3.10
conda activate bigdata
```

---

### Step 4 — Install Python Dependencies

```powershell
pip install pyspark==3.4.0 findspark jupyterlab pandas matplotlib
```

| Package | Purpose |
|---|---|
| `pyspark==3.4.0` | Apache Spark Python API |
| `findspark` | Locates PySpark automatically in Jupyter |
| `jupyterlab` | Notebook environment |
| `pandas` | Used for `.toPandas()` calls in visualisation cells |
| `matplotlib` | Charts in the batch analysis notebook |

---

### Step 5 — Verify the Environment

Run this quick check in a Python script or notebook cell:

```python
import os
import findspark

os.environ["JAVA_HOME"]   = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.30.7-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"]        = r"C:\hadoop\bin;" + os.environ.get("PATH", "")

findspark.init()

from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("test").getOrCreate()
spark.range(5).show()
spark.stop()
print("Environment OK")
```

Expected output:
```
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+
Environment OK
```

---

### Step 6 — Clone / Open the Project

```powershell
git clone https://github.com/miguel-tm/big-data_group-10.git
cd big-data_group-10
jupyter lab
```

Then run the notebooks **in order**:

| Order | Notebook / Script | Run once? | Description |
|---|---|---|---|
| 1 | `01_bronze_to_silver.ipynb` | Yes | ETL: raw CSV → canonical Silver Parquet |
| 2 | `02_batch_analysis.ipynb` | Yes | Batch Gold outputs + run summary |
| 3 | `src/make_replay_files.py` | Yes | Generates streaming replay chunks under `data/replay/` |
| 4 | `src/streaming_job.py` + `src/dropper.py` | Run together in two terminals | Live streaming simulation |
| 5 | `03_streaming_logic_preview.ipynb` | After streaming run | Validates streaming output correctness |
| 6 | `04_orchestration.ipynb` | Final | Experiment log and batch vs streaming comparison |

---

### Common Issues & Fixes

| Error | Cause | Fix |
|---|---|---|
| `winutils.exe not found` | `HADOOP_HOME` not set or wrong path | Verify `C:\hadoop\bin\winutils.exe` exists; check `os.environ["HADOOP_HOME"]` in the script |
| `JAVA_HOME not set` | Java not installed or env var missing | Complete Step 1; restart terminal and kernel |
| `Unable to infer schema for Parquet` | Reading empty or metadata-only directory | Ensure the upstream step ran and produced `.parquet` data files |
| `AnalysisException: path not found` | Relative path resolved from wrong working directory | Use absolute paths or the `PROJECT_ROOT` constant defined in each notebook |
| `.tmp file not found` during streaming | Dropper's in-flight temp file picked up by Spark scanner | `streaming_job.py` uses `.option("pathGlobFilter", "*.parquet")` to ignore `.tmp` files |

> **One-time pip install (if not using Anaconda):**
> ```powershell
> pip install pyspark==3.4.0 findspark jupyterlab pandas matplotlib
> ```

> **Why Hadoop/winutils?**
> Spark on Windows requires `winutils.exe` and `hadoop.dll` for local filesystem access (reading/writing Parquet, HDFS-style directory operations). Without them you will see `UnsatisfiedLinkError` or `NativeIO$Windows` errors. This is a Windows-only compatibility shim — no actual Hadoop cluster is needed.